In [ ]:
# ======================================================================================
# [전략 요약]
# 1. 핵심 가설: 특정 아이템(Leader)의 변화가 시차(Lag)를 두고 다른 아이템(Follower)에 영향을 준다.
# 2. 모델링: 선형 모델(Ridge)과 트리 모델(LGBM, CatBoost)을 결합하여 추세와 비선형 패턴을 모두 잡음.
# 3. 앙상블: Ridge에 60% 가중치를 주어 과적합을 방지하고 안정성을 확보.
# 4. 후처리: Zero-Cut(10.0 미만 절삭)을 통해 노이즈가 많은 미세 예측값을 제거하여 점수 향상 도모.
# ======================================================================================

# 0. 라이브러리 설치 및 임포트
# - CatBoost, LightGBM 등 부스팅 모델 라이브러리 설치
# - 데이터 핸들링(Pandas, Numpy) 및 모델링(Sklearn) 도구 불러오기
!pip install catboost lightgbm

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import lightgbm as lgb
from catboost import CatBoostRegressor
from tqdm import tqdm
import os

# 1. 데이터 로드
# - vs code 대신 코랩(Colab) 환경에 따라 경로를 자동 인식하여 train.csv 로드
if 'train.csv' in os.listdir():
    train = pd.read_csv('./train.csv')
else:
    train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')

# 2. 데이터 전처리 (Time Series Pivot)
# - 날짜 컬럼 생성 및 월 단위(ym) 변환
# - 데이터를 [아이템 x 월] 형태의 피벗 테이블로 변환하여 시계열 처리가 쉽도록 구조 변경
train['date'] = pd.to_datetime(train['year'].astype(str) + '-' + train['month'].astype(str).str.zfill(2))
train['ym'] = train['date'].dt.to_period('M')
item_hs_map = train[['item_id', 'hs4']].drop_duplicates().set_index('item_id') # 아이템별 HS코드 매핑 정보 저장
monthly = train.groupby(["item_id", "ym"], as_index=False)["value"].sum()
pivot = monthly.pivot(index="item_id", columns="ym", values="value").fillna(0.0)

# 3. 공행성(Comovement) 분석 - Leader & Follower 찾기
# - safe_corr: 표준편차가 0인 경우(변동 없음) 에러 방지를 위한 상관계수 계산 함수
# - find_comovement_pairs:
#   모든 아이템 쌍을 비교하여, 특정 시차(Lag 1~12)를 두었을 때 상관관계가 0.4 이상인 '리더-팔로워' 쌍을 발굴
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0: return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def find_comovement_pairs(pivot, max_lag=12, min_nonzero=12, corr_threshold=0.4):
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []
    # 모든 아이템을 순회하며 리더(Leader) 후보 선정
    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero: continue # 데이터가 너무 적으면 패스
        # 다른 아이템들과 비교하여 팔로워(Follower) 후보 찾기
        for follower in items:
            if follower == leader: continue
            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero: continue
            
            # 최적의 시차(Best Lag)와 상관계수 탐색
            best_lag, best_corr = None, 0.0
            for lag in range(1, max_lag + 1):
                if n_months <= lag: continue
                corr = safe_corr(x[:-lag], y[lag:]) # 시차를 적용하여 상관계수 계산
                if abs(corr) > abs(best_corr):
                    best_corr, best_lag = corr, lag
            
            # 상관계수가 기준치(0.4) 이상이면 유의미한 쌍으로 저장
            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({"leading_item_id": leader, "following_item_id": follower, "best_lag": best_lag, "max_corr": best_corr})
    return pd.DataFrame(results)

pairs = find_comovement_pairs(pivot, max_lag=12, min_nonzero=12, corr_threshold=0.4)

# 4. 학습 데이터셋 생성 (Feature Engineering)
# - 발굴된 Pair 정보를 기반으로 학습용 데이터 생성
# - 파생 변수 생성:
#   1) 타겟 아이템(Follower)의 과거 값 (Lag 1, 2, 6, 12개월 전)
#   2) 최근 3개월 이동 평균 (b_mean_3)
#   3) 리더 아이템(Leader)의 시차 적용 값 (a_t_lag) <- 핵심 피처!
#   4) 데이터 값은 로그 변환(np.log1p)하여 정규성 확보
def build_training_data(pivot, pairs, item_hs_map):
    months = pivot.columns.to_list()
    n_months = len(months)
    rows = []
    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)
        if leader not in pivot.index or follower not in pivot.index: continue
        follower_hs = item_hs_map.loc[follower, 'hs4']
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        
        start_idx = max(lag, 12) # 최소 1년치 데이터 확보 후 시작
        for t in range(start_idx, n_months - 1):
            if t < 18: continue # 안정적인 학습을 위해 초기 데이터 건너뜀
            
            # 로그 변환된 시계열 피처 생성
            b_t = np.log1p(b_series[t]) # 현재 시점
            b_t_1 = np.log1p(b_series[t - 1]) # 1달 전
            b_t_2 = np.log1p(b_series[t - 2]) # 2달 전
            b_t_12 = np.log1p(b_series[t - 12]) # 1년 전 (계절성 반영)
            b_t_6 = np.log1p(b_series[t - 6]) # 반년 전
            a_t_lag = np.log1p(a_series[t - lag]) # 리더의 Lag 시점 값
            b_mean_3 = (b_t + b_t_1 + b_t_2) / 3.0 # 이동 평균
            current_month = (t % 12) + 1 # 몇월인지 알려주는 값
            target = np.log1p(b_series[t + 1]) # 예측 목표: 다음 달 값
            
            rows.append({
                "b_t": b_t, "b_t_1": b_t_1, "b_mean_3": b_mean_3, "b_t_6": b_t_6, "b_t_12": b_t_12,
                "a_t_lag": a_t_lag, "max_corr": corr, "best_lag": float(lag), 
                "month": current_month, "item_id": follower, "hs4": follower_hs,
                "target": target, "time_index": t 
            })
    return pd.DataFrame(rows)

df_train_model = build_training_data(pivot, pairs, item_hs_map)

# --- 모델별 데이터셋 분리 ---
# Ridge(선형 모델)용: 범주형 변수(월, 아이템ID, HS코드)를 원-핫 인코딩(One-Hot Encoding) 처리
df_train_encoded = pd.get_dummies(df_train_model, columns=['month', 'item_id', 'hs4'], prefix=['month', 'item', 'hs'])
ridge_features = [col for col in df_train_encoded.columns if col not in ['target', 'leading_item_id', 'following_item_id']]
X_ridge = df_train_encoded[ridge_features].values
y_ridge = df_train_encoded["target"].values
w_ridge = df_train_encoded["time_index"].values ** 2 # 최신 데이터에 가중치 부여 (Time Decay)

# Tree 모델(LGBM, CatBoost)용: 원-핫 인코딩 없이 원본 피처 사용
tree_features = ['b_t', 'b_t_1', 'b_mean_3', 'b_t_6', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month', 'hs4', 'time_index']
X_tree = df_train_model[tree_features].copy()
y_tree = df_train_model["target"]

# LGBM용 범주형 변수 타입 변환
X_lgb = X_tree.copy()
for col in ['month', 'hs4']: X_lgb[col] = X_lgb[col].astype('category')
cat_features_indices = ['month', 'hs4'] # CatBoost용 범주형 인덱스 지정

# 5. 모델 학습 (3가지 모델 앙상블 준비)
print("1. Ridge (Alpha=10.0) 학습 중... (선형 추세 포착)")
model_ridge = Ridge(alpha=10.0)
model_ridge.fit(X_ridge, y_ridge, sample_weight=w_ridge)

print("2. LightGBM (Default) 학습 중... (빠른 속도, 대용량 데이터 강점)")
train_data_lgb = lgb.Dataset(X_lgb, label=y_tree, weight=w_ridge)
params_lgb = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.9, 
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'verbose': -1, 'seed': 42
}
model_lgb = lgb.train(params_lgb, train_data_lgb, num_boost_round=1000)

print("3. CatBoost (Super Mode) 학습 중... (범주형 데이터 처리에 강점)")
model_cb = CatBoostRegressor(
    iterations=2000, learning_rate=0.03, depth=10, loss_function='RMSE',
    verbose=0, cat_features=cat_features_indices, random_seed=42
)
model_cb.fit(X_tree, y_tree, sample_weight=w_ridge)

# 6. 최종 예측 및 앙상블 (Zero-Cut 전략 포함)
# - 테스트 시점(마지막 달) 데이터를 기반으로 피처를 생성하고 예측 수행
def predict_final_zerocut_tuned(pivot, pairs, model_ridge, model_lgb, model_cb, item_hs_map, ridge_cols, weights, zero_cut=5.0):
    # (시간 인덱스 계산 로직 생략 - 마지막 시점 t_last 기준)
    months = pivot.columns.to_list()
    n_months = len(months)
    t_last = n_months - 1
    # ... (Lag 시점 계산 변수들) ...
    t_prev = n_months - 2
    t_prev_2 = n_months - 3
    t_minus_12 = n_months - 1 - 12
    t_minus_6 = n_months - 1 - 6
    last_month = (t_last % 12) + 1
    
    # 가중치 정규화 (Ridge : LGBM : CatBoost)
    w1, w2, w3 = weights
    total_w = w1 + w2 + w3
    w1, w2, w3 = w1/total_w, w2/total_w, w3/total_w
    
    preds = []
    # 각 Pair 별로 예측 수행
    for row in tqdm(pairs.itertuples(index=False)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)
        if leader not in pivot.index or follower not in pivot.index: continue
        follower_hs = item_hs_map.loc[follower, 'hs4']
        
        # (테스트용 피처 생성: b_t, b_t_1, a_t_lag 등 학습 때와 동일한 로직)
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        if t_last - lag < 0: continue
        
        b_t = np.log1p(b_series[t_last])
        b_t_1 = np.log1p(b_series[t_prev])
        b_t_2 = np.log1p(b_series[t_prev_2])
        a_t_lag = np.log1p(a_series[t_last - lag])
        b_mean_3 = (b_t + b_t_1 + b_t_2) / 3.0
        b_t_12 = np.log1p(b_series[t_last - 12]) if t_minus_12 >= 0 else 0.0
        b_t_6 = np.log1p(b_series[t_last - 6]) if t_minus_6 >= 0 else 0.0
        
        # 1) Ridge 예측 (수동으로 One-Hot 벡터 생성하여 입력)
        input_dict = {'b_t': b_t, 'b_t_1': b_t_1, 'b_mean_3': b_mean_3, 'b_t_6': b_t_6, 'b_t_12': b_t_12, 'a_t_lag': a_t_lag, 'max_corr': corr, 'best_lag': float(lag), 'time_index': t_last}
        ridge_input = []
        for col in ridge_cols:
            if col in input_dict: ridge_input.append(input_dict[col])
            elif col == f'month_{last_month}': ridge_input.append(1)
            elif col == f'item_{follower}': ridge_input.append(1)
            elif col == f'hs_{follower_hs}': ridge_input.append(1)
            else: ridge_input.append(0)
        pred_ridge = model_ridge.predict([ridge_input])[0]
        
        # 2) Tree (LGBM, CatBoost) 예측
        tree_input_df = pd.DataFrame([{
            'b_t': b_t, 'b_t_1': b_t_1, 'b_mean_3': b_mean_3, 'b_t_6': b_t_6, 'b_t_12': b_t_12,
            'a_t_lag': a_t_lag, 'max_corr': corr, 'best_lag': float(lag), 
            'month': last_month, 'hs4': follower_hs, 'time_index': t_last
        }])
        
        lgb_input = tree_input_df.copy()
        for col in ['month', 'hs4']: lgb_input[col] = lgb_input[col].astype('category')
        pred_lgb = model_lgb.predict(lgb_input)[0]
        pred_cb = model_cb.predict(tree_input_df)[0]
        
        # 3) 가중 평균 앙상블 (Ridge: 60%, LGBM: 20%, Cat: 20%)
        # - Ridge 비중을 높여 일반화 성능을 강조함
        final_log_pred = (pred_ridge * w1) + (pred_lgb * w2) + (pred_cb * w3)
        
        # 로그 역변환 (exp)
        y_pred = int(round(max(0.0, float(np.expm1(final_log_pred)))))
        
        # [핵심 후처리] Zero Cut (5.0 -> 10.0 상향)
        # - 예측값이 10 미만인 경우 0으로 강제 변환하여 노이즈 제거
        if y_pred < zero_cut:
            y_pred = 0
            
        preds.append({"leading_item_id": leader, "following_item_id": follower, "value": y_pred})
        
    return pd.DataFrame(preds)

# 앙상블 가중치 설정 (Ridge : LGBM : Cat = 0.6 : 0.2 : 0.2)
weights = (0.6, 0.2, 0.2)

# 최종 예측 실행 및 CSV 저장 (Zero Cut 기준값 10.0 적용)
submission = predict_final_zerocut_tuned(pivot, pairs, model_ridge, model_lgb, model_cb, item_hs_map, ridge_features, weights, zero_cut=10.0)
submission.to_csv('./submission_final_zerocut10.csv', index=False)
print("Zero Cut(10.0) + 최고점 앙상블 저장 완료!")